In [1]:
import pandas as pd

In [41]:
orders = pd.read_csv(r"D:\Reading Material\Python\Practice Questions\Notebook\Projects\Automation Ideas\Reconcillation Engine\Ecommerce Dataset\Ecommerce Order Dataset\train\Orders.csv")
payments = pd.read_csv(r"D:\Reading Material\Python\Practice Questions\Notebook\Projects\Automation Ideas\Reconcillation Engine\Ecommerce Dataset\Ecommerce Order Dataset\train\Payments.csv")
order_items = pd.read_csv(r"D:\Reading Material\Python\Practice Questions\Notebook\Projects\Automation Ideas\Reconcillation Engine\Ecommerce Dataset\Ecommerce Order Dataset\train\OrderItems.csv")


In [42]:
orders.columns.to_list()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_timestamp',
 'order_estimated_delivery_date']

In [43]:
payments.columns.to_list()

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [44]:
order_items.columns.to_list()

['order_id', 'product_id', 'seller_id', 'price', 'shipping_charges']

In [45]:
# Aggregate order items → total order value
order_items["order_value"] = (order_items["price"] + order_items["shipping_charges"])
order_totals = order_items.groupby("order_id", as_index=False)["order_value"].sum()

In [46]:
# Aggregate payments → total payment per order
payment_totals = (payments.groupby("order_id", as_index=False)["payment_value"].sum())

In [47]:
# Merge the Two Tables
recon = pd.merge(order_totals, payment_totals, on="order_id", how="outer")

# Calculate Variance #Variance tells us: Did the customer pay fully? Did they underpay? Did they overpay?
recon["variance"] = (recon["payment_value"] - recon["order_value"])

In [48]:
# Check status of payment

def reconciliation_status(row):
    if pd.isna(row["order_value"]):
        return "Payment Without Order"

    elif pd.isna(row["payment_value"]):
        return "Order Without Payment"

    elif abs(row["variance"]) < 0.01:
        return "Matched"

    elif row["variance"] > 0:
        return "Overpayment"

    else:
        return "Underpayment"

#merged["reconciliation_status"] = merged.apply(
#    lambda row: "Match" if row["total_item_value"] == row["payment_value"] else "Mismatch",
#    axis=1)

In [49]:
# "Take each order one by one, send it to the reconciliation function, get the result, 
# and store it in a new column called status."
recon["status"] = recon.apply(
    reconciliation_status,
    axis=1)

# Revenue Exposure
recon["revenue_exposure"] = recon["variance"].abs()

In [50]:
# Join with orders for customer/timestamp context
final_report = recon.merge(
    orders[
        ["order_id", "customer_id", "order_status", "order_purchase_timestamp"]
    ],
    on="order_id",
    how="left"
)

In [51]:
# Summary stats
summary = pd.DataFrame({

    "Metric": [
        "Total Orders",
        "Matched Orders",
        "Underpayments",
        "Overpayments",
        "Missing Payments",
        "Revenue Exposure"
    ],

    "Value": [

        len(final_report),

        (final_report["status"] == "Matched").sum(),

        (final_report["status"] == "Underpayment").sum(),

        (final_report["status"] == "Overpayment").sum(),

        (final_report["status"] == "Order Without Payment").sum(),

        final_report["revenue_exposure"].sum()
    ]
})

In [52]:
# Export mismatches as Exception Sheets
matched = final_report[
    final_report["status"] == "Matched"]

underpaid = final_report[
    final_report["status"] == "Underpayment"]

overpaid = final_report[
    final_report["status"] == "Overpayment"]

missing = final_report[
    final_report["status"] == "Order Without Payment"]

In [53]:
# Export all data to Excel
with pd.ExcelWriter("Revenue_Reconciliation.xlsx", engine="openpyxl") as writer:
    matched.to_excel(writer, sheet_name="Matched", index=False)
    underpaid.to_excel(writer, sheet_name="Underpayments", index=False)
    overpaid.to_excel( writer, sheet_name="Overpayments", index=False)
    missing.to_excel(writer, sheet_name="Missing Payments", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)